In [ ]:

import os, sys, glob, shutil, time, json
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

Xh=np.load(prev+"/features_human.npy"); E1=np.load(prev+"/features_eval_lex.npy")
E2=np.load(prev+"/features_eval_mixed.npy"); Xl=np.load(prev+"/features_llm.npy")
log(f"признаки: ручные {Xh.shape}, LLM {Xl.shape}, линейки {E1.shape}/{E2.shape}")
items=pd.read_parquet(base+"/items_human.parquet",columns=["id","category"])
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
cat_of=dict(zip(items["id"],items["category"].astype(str)))
cp=hm["id1"].map(cat_of).astype(str).to_numpy()
y=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
tr,va=np.flatnonzero(tm),np.flatnonzero(vm); rel=np.flatnonzero(~vm)
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,yy): return float(np.mean([average_precision_score(yy[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(yy[c==k]))>1]))
P=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,random_state=0,early_stopping=False)
def run(tag,X,yy):
    c=HistGradientBoostingClassifier(**P).fit(X,yy)
    ph=c.predict_proba(Xh[va])[:,1]; p1=c.predict_proba(E1)[:,1]; p2=c.predict_proba(E2)[:,1]
    np.save(f"/kaggle/working/pred_{tag}_lex.npy",p1); np.save(f"/kaggle/working/pred_{tag}_mix.npy",p2)
    log(f"{tag:<22} holdout {macro_pr_auc(y[va],ph,cp[va])[0]:.6f}  лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
run("ручные_строгие",Xh[tr],y[tr])
run("ручные_расширенные",Xh[rel],y[rel])
run("LLM",Xl,yl)
run("LLM+ручные",np.vstack([Xl,Xh[rel]]),np.concatenate([yl,y[rel]]))
w=np.concatenate([np.full(len(Xl),1.0),np.full(len(rel),3.0)])
c=HistGradientBoostingClassifier(**P).fit(np.vstack([Xl,Xh[rel]]),np.concatenate([yl,y[rel]]),sample_weight=w)
p1=c.predict_proba(E1)[:,1]; p2=c.predict_proba(E2)[:,1]
np.save("/kaggle/working/pred_смесь3к1_lex.npy",p1); np.save("/kaggle/working/pred_смесь3к1_mix.npy",p2)
log(f"{'LLM+ручные вес 3':<22} holdout {macro_pr_auc(y[va],c.predict_proba(Xh[va])[:,1],cp[va])[0]:.6f}  "
    f"лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
log("готово")
